## 2. Дерево решений

### Подготовим признаки для первого дерева

In [1]:
%%time
# Задание 2 — загрузка выборок и выделение целевой переменной
import warnings

warnings.filterwarnings("ignore")

from pathlib import Path

import pandas as pd

RANDOM_STATE = 21
DATA_DIR = Path("data")
TARGET = "IsBadBuy"

df_train = pd.read_csv(DATA_DIR / "df_train.csv")
df_valid = pd.read_csv(DATA_DIR / "df_valid.csv")

X_train = df_train.drop(columns=[TARGET])
y_train = df_train[TARGET]

X_valid = df_valid.drop(columns=[TARGET])
y_valid = df_valid[TARGET]


CPU times: total: 984 ms
Wall time: 1 s


### закодируем категории и заполним пропуски, используя статистики только из train

In [2]:
%%time
# Задание 2 — кодирование категорий и заполнение пропусков
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

categorical_columns = [
    "Auction", "Make", "Model", "Trim", "SubModel", "Color",
    "Transmission", "WheelType", "Nationality", "Size",
    "TopThreeAmericanName", "BYRNO", "VNZIP1", "VNST",
]

excluded_columns = {
    "RefId", "PurchDate", "WheelTypeID",
    "PRIMEUNIT", "AUCGUART", "IsOnlineSale",
}

numeric_columns = [
    column for column in X_train.columns
    if column not in categorical_columns and column not in excluded_columns
]

categorical_pipeline = make_pipeline(
    SimpleImputer(strategy="constant", fill_value="__MISSING__"),
    OneHotEncoder(handle_unknown="ignore"),
)

preprocessor = ColumnTransformer(
    [
        ("categorical", categorical_pipeline, categorical_columns),
        ("numeric", SimpleImputer(strategy="median"), numeric_columns),
    ],
    sparse_threshold=1.0,
)

X_train_encoded = preprocessor.fit_transform(X_train)
X_valid_encoded = preprocessor.transform(X_valid)

assert X_train_encoded.shape[0] == len(y_train)
assert X_valid_encoded.shape[0] == len(y_valid)
assert X_train_encoded.shape[1] == X_valid_encoded.shape[1]

print(f"Train: {X_train_encoded.shape}, {type(X_train_encoded).__name__}")
print(f"Validation: {X_valid_encoded.shape}, {type(X_valid_encoded).__name__}")

Train: (24391, 1857), csr_matrix
Validation: (24255, 1857), csr_matrix
CPU times: total: 1.56 s
Wall time: 1.56 s


### Реализуем узел дерева Node. Узел будет хранить индексы своих строк, долю объектов класса 1 и ссылки на дочерние узлы.

In [3]:
%%time
# Задание 2 — класс Node и неоднородность Джини
import numpy as np


class Node:
    def __init__(self, row_indices, y):
        self.row_indices = row_indices
        self.left = None
        self.right = None

        self.n_samples = len(row_indices)
        self.n_positive = int(np.count_nonzero(y[row_indices] == 1))
        self.probability = self.n_positive / self.n_samples

    def gini(self):
        p = self.probability
        return 1.0 - p ** 2 - (1.0 - p) ** 2


y_train_array = y_train.to_numpy()


CPU times: total: 0 ns
Wall time: 0 ns


### Найдём лучшее разбиение по одному признаку. Функция примет значения одного столбца и соответствующие метки 0 и 1.

In [4]:
%%time
# Задание 2 — поиск порога по критерию Джини

def best_gini_split(values, labels):
    order = np.argsort(values)
    sorted_values = values[order]
    sorted_labels = labels[order]

    cuts = np.flatnonzero(sorted_values[:-1] < sorted_values[1:])
    if cuts.size == 0:
        return None, None

    left_size = cuts + 1
    right_size = len(values) - left_size

    positive_prefix = np.cumsum(sorted_labels == 1)
    left_positive = positive_prefix[cuts]
    right_positive = positive_prefix[-1] - left_positive

    p_left = left_positive / left_size
    p_right = right_positive / right_size

    gini_left = 2 * p_left * (1 - p_left)
    gini_right = 2 * p_right * (1 - p_right)
    weighted_gini = (
                            left_size * gini_left + right_size * gini_right
                    ) / len(values)

    best = np.argmin(weighted_gini)
    cut = cuts[best]
    threshold = (sorted_values[cut] + sorted_values[cut + 1]) / 2

    return float(threshold), float(weighted_gini[best])


CPU times: total: 0 ns
Wall time: 0 ns


### 2.1 DecisionTreeClassifier принимает max_depth в конструкторе и сохраняет его в self.max_depth:

In [5]:
%%time
# Задание 2 — класс DecisionTreeClassifier и обучение дерева

class DecisionTreeClassifier:
    def __init__(self, max_depth=7):
        self.max_depth = max_depth

    def fit(self, X, y):
        X_csc = X.tocsc()
        labels = np.asarray(y)
        rows = np.arange(X_csc.shape[0])
        self.root_ = self._grow(X_csc, labels, rows, depth=0)
        return self

    def _grow(self, X, y, rows, depth):
        node = Node(rows, y)
        node.feature = None
        node.threshold = None

        if depth >= self.max_depth or node.gini() == 0.0:
            return node

        feature, threshold, score = self._best_node_split(X, y, rows)

        if feature is None or node.gini() - score <= 1e-12:
            return node

        values = X[rows, feature].toarray().ravel()
        left_mask = values <= threshold

        # 2.2 	Node содержит self.left и self.right. При создании они равны None, а при разбиении _grow записывает в них дочерние узлы.

        node.feature = feature
        node.threshold = threshold
        node.left = self._grow(X, y, rows[left_mask], depth + 1)
        node.right = self._grow(X, y, rows[~left_mask], depth + 1)
        return node

    # ==================================================================================================

    def predict_proba(self, X):
        X_csc = X.tocsc()
        positive = np.empty(X_csc.shape[0], dtype=float)

        def fill_probabilities(node, rows):
            if rows.size == 0:
                return

            if node.feature is None:
                positive[rows] = node.probability
                return

            values = X_csc[rows, node.feature].toarray().ravel()
            left_mask = values <= node.threshold

            fill_probabilities(node.left, rows[left_mask])
            fill_probabilities(node.right, rows[~left_mask])

        fill_probabilities(self.root_, np.arange(X_csc.shape[0]))
        return np.column_stack((1.0 - positive, positive))

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] > 0.5).astype(int)

    # ==================================================================================================
    # 	2.3
    # 	Метод DecisionTreeClassifier._best_node_split() выбирает разбиение
    # 	с минимальной взвешенной неоднородностью среди всех признаков.
    # 	При построении дерева _grow вызывает этот метод через self.
    #   Эта функция находит наилучшее возможное разделение в текущем узле
    def _best_node_split(self, X, y, row_indices):
        node_labels = y[row_indices]
        best_feature = None
        best_threshold = None
        best_score = np.inf

        for feature in range(X.shape[1]):
            values = X[row_indices, feature].toarray().ravel()
            threshold, score = best_gini_split(values, node_labels)

            if score is not None and score < best_score:
                best_feature = feature
                best_threshold = threshold
                best_score = score

        return best_feature, best_threshold, best_score


# ==================================================================================================

stump = DecisionTreeClassifier(max_depth=1).fit(
    X_train_encoded, y_train_array
)

CPU times: total: 2.09 s
Wall time: 2.11 s


In [11]:
%%time
# Задание 2 — поиск порога для дерева регрессии
def best_regression_split(values, targets):
    order = np.argsort(values)
    sorted_values = values[order]
    sorted_targets = np.asarray(targets, dtype=float)[order]

    cuts = np.flatnonzero(sorted_values[:-1] < sorted_values[1:])
    if cuts.size == 0:
        return None, None

    left_size = cuts + 1
    right_size = len(values) - left_size

    # Центрирование помогает точнее вычислить дисперсию.
    centered = sorted_targets - sorted_targets.mean()
    prefix_sum = np.cumsum(centered)
    prefix_square_sum = np.cumsum(centered**2)

    left_sum = prefix_sum[cuts]
    right_sum = prefix_sum[-1] - left_sum
    left_square_sum = prefix_square_sum[cuts]
    right_square_sum = prefix_square_sum[-1] - left_square_sum

    left_variance = left_square_sum / left_size - (left_sum / left_size)**2
    right_variance = right_square_sum / right_size - (right_sum / right_size)**2

    left_std = np.sqrt(np.maximum(left_variance, 0.0))
    right_std = np.sqrt(np.maximum(right_variance, 0.0))
    scores = (left_size * left_std + right_size * right_std) / len(values)

    best = np.argmin(scores)
    cut = cuts[best]
    low = float(sorted_values[cut])
    high = float(sorted_values[cut + 1])
    threshold = low / 2 + high / 2

    return threshold, float(scores[best])

CPU times: total: 0 ns
Wall time: 0 ns


In [7]:
%%time
# Задание 2 — класс DecisionTreeRegressor и проверка на примере

from scipy.sparse import csc_matrix


class RegressionNode:
    def __init__(self, targets):
        self.prediction = float(np.mean(targets))
        self.feature = None
        self.threshold = None
        self.left = None
        self.right = None


class DecisionTreeRegressor:
    def __init__(self, max_depth=7):
        self.max_depth = max_depth

    def fit(self, X, y):
        X_csc = csc_matrix(X)
        targets = np.asarray(y, dtype=float)
        rows = np.arange(X_csc.shape[0])
        self.root_ = self._grow(X_csc, targets, rows, depth=0)
        return self

    def _best_node_split(self, X, targets, rows):
        node_targets = targets[rows]
        best_feature = None
        best_threshold = None
        best_score = np.inf

        for feature in range(X.shape[1]):
            values = X[rows, feature].toarray().ravel()
            threshold, score = best_regression_split(values, node_targets)

            if score is not None and score < best_score:
                best_feature = feature
                best_threshold = threshold
                best_score = score

        return best_feature, best_threshold, best_score

    def _grow(self, X, targets, rows, depth):
        node_targets = targets[rows]
        node = RegressionNode(node_targets)

        if depth >= self.max_depth or np.all(node_targets == node_targets[0]):
            return node

        feature, threshold, score = self._best_node_split(X, targets, rows)
        if feature is None or score >= node_targets.std():
            return node

        values = X[rows, feature].toarray().ravel()
        left_mask = values <= threshold

        node.feature = feature
        node.threshold = threshold
        node.left = self._grow(X, targets, rows[left_mask], depth + 1)
        node.right = self._grow(X, targets, rows[~left_mask], depth + 1)
        return node

    def predict(self, X):
        X_csc = csc_matrix(X)
        predictions = np.empty(X_csc.shape[0], dtype=float)

        def fill_predictions(node, rows):
            if rows.size == 0:
                return

            if node.feature is None:
                predictions[rows] = node.prediction
                return

            values = X_csc[rows, node.feature].toarray().ravel()
            left_mask = values <= node.threshold
            fill_predictions(node.left, rows[left_mask])
            fill_predictions(node.right, rows[~left_mask])

        fill_predictions(self.root_, np.arange(X_csc.shape[0]))
        return predictions


values_example = np.array([30, 10, 40, 20, 20])
targets_example = np.array([9.0, 2.0, 9.0, 2.0, 2.0])

X_example = values_example.reshape(-1, 1)
reg_tree = DecisionTreeRegressor(max_depth=1).fit(X_example, targets_example)
reg_predictions = reg_tree.predict(X_example)

assert reg_tree.root_.feature == 0
assert np.isclose(reg_tree.root_.threshold, 25.0)
assert np.allclose(reg_predictions, targets_example)

print(f"Порог в корне: {reg_tree.root_.threshold:g}")
print(f"Предсказания: {reg_predictions}")

Порог в корне: 25
Предсказания: [9. 2. 9. 2. 2.]
CPU times: total: 0 ns
Wall time: 1.22 ms


Дерево выбрало порог 25, чтобы разделить примеры на две группы. При значении признака до 25 включительно оно предсказывает 2, а при значении выше 25 предсказывает 9. В каждой группе предсказание равно среднему целевому значению.

Для пяти точек в исходном порядке получились предсказания 9, 2, 9, 2, 2. Они полностью совпали с целевыми значениями учебного примера.

In [8]:
%%time
# Задание 2 — Extra Randomized Tree и проверка воспроизводимости

class ExtraRandomizedTreeClassifier(DecisionTreeClassifier):
    def __init__(self, max_depth=7, random_state=21):
        super().__init__(max_depth=max_depth)
        self.random_state = random_state

    def fit(self, X, y):
        self._rng = np.random.default_rng(self.random_state)
        return super().fit(X, y)

    def _best_node_split(self, X, y, row_indices):
        node_labels = y[row_indices]
        best_feature = None
        best_threshold = None
        best_score = np.inf

        for feature in range(X.shape[1]):
            values = X[row_indices, feature].toarray().ravel()
            low, high = values.min(), values.max()
            if low == high:
                continue

            threshold = self._rng.uniform(low, high)
            left_mask = values <= threshold
            right_mask = ~left_mask
            if not left_mask.any() or not right_mask.any():
                continue

            n_left = left_mask.sum()
            n_right = right_mask.sum()
            p_left = node_labels[left_mask].mean()
            p_right = node_labels[right_mask].mean()

            gini_left = 2 * p_left * (1 - p_left)
            gini_right = 2 * p_right * (1 - p_right)
            score = (n_left * gini_left + n_right * gini_right) / len(row_indices)

            if score < best_score:
                best_feature = feature
                best_threshold = threshold
                best_score = score

        return best_feature, best_threshold, best_score


X_extra_example = csc_matrix([[0], [1], [2], [3], [4]])
y_extra_example = np.array([0, 0, 0, 1, 1])

extra_tree = ExtraRandomizedTreeClassifier(max_depth=1, random_state=21)
extra_tree.fit(X_extra_example, y_extra_example)
first_threshold = extra_tree.root_.threshold
first_proba = extra_tree.predict_proba(X_extra_example)

assert extra_tree.root_.feature == 0
assert 0 <= first_threshold < 4

extra_tree.fit(X_extra_example, y_extra_example)
assert extra_tree.root_.threshold == first_threshold
assert np.array_equal(extra_tree.predict_proba(X_extra_example), first_proba)

print(f"Случайный порог в корне: {first_threshold:.6f}")
print("Порог и вероятности воспроизводятся при повторном обучении.")

Случайный порог в корне: 3.124470
Порог и вероятности воспроизводятся при повторном обучении.
CPU times: total: 0 ns
Wall time: 2.06 ms


Дерево случайно выбрало порог 3,124470 для корневого узла. Значения признака 0, 1, 2 и 3 попали в левый лист, а значение 4 попало в правый. Вероятность класса 1 в каждом листе равна доле объектов этого класса: 0,25 слева и 1,0 справа.

При повторном обучении на тех же данных с random_state=21 получились тот же порог и те же вероятности. Проверка воспроизводимости прошла.


### 3.1 Самописное дерево решений глубины 1:

In [9]:
%%time
# Задание 3 — ROC AUC и Gini на валидации
from sklearn.metrics import roc_auc_score

valid_proba = stump.predict_proba(X_valid_encoded)

assert valid_proba.shape == (len(y_valid), 2)
assert np.all((valid_proba >= 0) & (valid_proba <= 1))
assert np.allclose(valid_proba.sum(axis=1), 1.0)

valid_auc = roc_auc_score(y_valid, valid_proba[:, 1])
valid_gini = 2 * valid_auc - 1

print(f"ROC AUC на валидации: {valid_auc:.14f}")
print(f"Gini на валидации: {valid_gini:.14f}")

ROC AUC на валидации: 0.61672530985506
Gini на валидации: 0.23345061971011
CPU times: total: 15.6 ms
Wall time: 15.8 ms


ROC AUC = 0,6167 и Gini = 0,2335 на валидационной выборке. Gini превышает требуемый порог 0,10.

## 4. Сравнение со Склёрном

In [14]:
%%time
# Задание 4. Сравнение со sklearn при max_depth=1

from sklearn.tree import DecisionTreeClassifier as SklearnDecisionTreeClassifier

sklearn_stump = SklearnDecisionTreeClassifier(
    criterion="gini",
    max_depth=1,
    random_state=21,
)
sklearn_stump.fit(X_train_encoded, y_train)

sklearn_valid_proba = sklearn_stump.predict_proba(X_valid_encoded)[:, 1]
sklearn_valid_gini = 2 * roc_auc_score(y_valid, sklearn_valid_proba) - 1

print(f"Самописное дерево: Gini = {valid_gini:.14f}")
print(f"Дерево sklearn:    Gini = {sklearn_valid_gini:.14f}")
print(f"Корень sklearn: признак {sklearn_stump.tree_.feature[0]}, "
      f"порог {sklearn_stump.tree_.threshold[0]:g}")
print(f"Вероятности совпадают: "
      f"{np.allclose(valid_proba[:, 1], sklearn_valid_proba)}")

Самописное дерево: Gini = 0.23345061971011
Дерево sklearn:    Gini = 0.23345061971011
Корень sklearn: признак 1647, порог 0.5
Вероятности совпадают: True
CPU times: total: 141 ms
Wall time: 155 ms


In [15]:
%%time
# Задание 4. Сравнение со sklearn при max_depth=3

depth = 3
models = {
    "Самописное дерево": DecisionTreeClassifier(max_depth=depth),
    "Дерево sklearn": SklearnDecisionTreeClassifier(
        criterion="gini",
        max_depth=depth,
        random_state=21,
    ),
}

for name, model in models.items():
    model.fit(X_train_encoded, y_train_array)

    train_scores = model.predict_proba(X_train_encoded)[:, 1]
    valid_scores = model.predict_proba(X_valid_encoded)[:, 1]

    gini_train = 2 * roc_auc_score(y_train, train_scores) - 1
    gini_valid = 2 * roc_auc_score(y_valid, valid_scores) - 1

    print(f"{name}: Gini train = {gini_train:.14f}, "
          f"Gini validation = {gini_valid:.14f}")

Самописное дерево: Gini train = 0.45343031997957, Gini validation = 0.42773005481532
Дерево sklearn: Gini train = 0.45343031997957, Gini validation = 0.42773005481532
CPU times: total: 9.22 s
Wall time: 9.78 s


На одинаковых выборках и при одинаковых настройках max_depth и criterion="gini" дерево из sklearn и самописное дерево показали одинаковый Gini при глубинах 1 и 3. При глубине 1 также совпали разбиение корня и предсказанные вероятности. При глубине 3 Gini на валидации на 0,0257 ниже, чем на обучении.